### Constants

In [21]:
DENOMINATOR: int = 400
K_FACTOR:    int = 16
# Could have variable k depending on games played, but if we have enough games constant is fine.
# Perhaps have variable depending on freshness of the team??

### Prediction and Update Formulae

Outcomes are given as a number from $0$ to $1$, where $0$ means the player/team on the left lost whilst $1$ means they won. If the outcome is $0.5$, we have a draw, and any number in between represents a win probability.

In [20]:
def expected_outcome(elo_a: float, elo_b: float) -> float:
    rb_ra: float = (elo_b - elo_a) / DENOMINATOR
    return 1 / (1 + 10 ** rb_ra)

In [22]:
def new_elos(elo_a: float, elo_b: float, outcome: float) -> tuple[float, float]:
    diff: float = K_FACTOR * (outcome - expected_outcome(elo_a, elo_b))
    return elo_a + diff, elo_b - diff

### Load Data

First cell formats the raw data and extracts the teams, second cell splits data into training and test.

In [123]:
from pandas import DataFrame
import pandas as pd

raw: DataFrame = pd.read_csv("data/nba_team_reference.csv")
raw["delta_points"] = raw["pts_home"] - raw["pts_away"]
raw.drop(columns=["season_year", "team_id_home", "team_id_away", "pts_home", "pts_away"], inplace=True)
raw.rename(
    columns={"game_id": "id", "game_date": "date", "team_abbreviation_home": "home", "team_abbreviation_away": "away", "home_win": "outcome"},
    inplace=True
)

teams: dict[str, float] = dict((team, 1000) for team in set(raw["home"].tolist() + raw["away"].tolist()))

assert len(teams) == 30

In [125]:
training: DataFrame = raw.sample(4 * len(raw) // 5).sort_values(by=["date"])
test:     DataFrame = raw[~raw["id"].isin(training["id"])].sort_values(by=["date"])

assert len(training) + len(test) == len(raw)
assert len(test["id"].isin(training["id"]).value_counts()) == 1

### Train

Go over each game in order and update team elos.

## CONSIDER DELTA POINTS!!!

In [136]:
for row in training.itertuples():
    new_home, new_away = new_elos(teams[row.home], teams[row.away], row.outcome)
    teams[row.home] = new_home
    teams[row.away] = new_away

# Teams by Elo
print([ team[0] for team in sorted(teams.items(), key=lambda item: item[1], reverse=True) ])

['BOS', 'OKC', 'CLE', 'IND', 'NYK', 'LAC', 'DEN', 'MIN', 'HOU', 'GSW', 'MIL', 'LAL', 'CHI', 'ORL', 'MEM', 'ATL', 'SAC', 'DAL', 'MIA', 'DET', 'PHX', 'POR', 'SAS', 'TOR', 'BKN', 'NOP', 'PHI', 'UTA', 'WAS', 'CHA']


### Test

In [140]:
predictions = test.copy()
predictions["prob"] = predictions.apply(lambda row: expected_outcome(teams[row["home"]], teams[row["away"]]), axis=1)
predictions["prediction"] = predictions["prob"].round()
predictions["correct"] = predictions["outcome"] == predictions["prediction"]

In [149]:
success = predictions["correct"].value_counts().to_dict()
print(success[True] / (success[True] + success[False]))

0.5346534653465347
